# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup

Import required packages

In [2]:
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path


Get an API key from [US Bureau of Labour Statistics API registration](https://data.bls.gov/registrationEngine/), and save it in the `.env` file in this folder.

The cell below reads that file and loads the key into a variable.

In [3]:
load_dotenv()

BLS_API_KEY = os.getenv("BLS_API_KEY")

## Load AI industry exposure index
In order to assess the differences in AI impacts across industries, I use AI Industry Exposure scores from "Occupational, Industry, and Geographic Exposure to Artificial Intelligence: A Novel Dataset and Its Potential" by Felten, Raj, and Seamans. The data annex containing these scores is available on Github: https://github.com/AIOE-Data/AIOE.


In [13]:
aiie = pd.read_excel(
    "https://github.com/AIOE-Data/AIOE/raw/refs/heads/main/AIOE_DataAppendix.xlsx", 
    sheet_name="Appendix B"
    )
aiie.columns = aiie.columns.str.lower().str.replace(" ", "_")
aiie.head()

,naics,industry_title,aiie
0,1133,Logging,-1.360161
1,1151,Support Activities for Crop Production,-2.165304
2,1152,Support Activities for Animal Production,-1.149307
3,2111,Oil and Gas Extraction,0.668615
4,2121,Coal Mining,-1.146242


There are multiple transformations that must be applied to the AIIE table in order to map the NAICS codes onto BLS series IDs:
- Several rows have NAICS codes ending in `0`, which is not a valid 4-digit level code. In some cases this reflects the AIIE score applying to multiple 4-digit codes listed in parentheses in the `industry_title`. I unpack these 4-digit codes into their own rows and apply the same AIIE score to them all in order to preserve the variation of AIIE scores as much as possible (aggregating these to the 3-digit level is tempting but would involve averaging out significant variation in AIIE scores in some instances, e.g. `4240: Merchant Wholesalers, Nondurable Goods`). 
- Rows with NAICS codes ending in `0` where 4-digit codes are not identified are treated as 3-digit level aggregates (e.g. `5310: Real Estate`). The dataset does not contain any such instances where the 3-digit level would overlap with another row at 4-digit level, so this mismatch of aggregation level will not cause any double counting.
- The NAICS codes which will be used to map industry AI exposure scores to BLS data series were updated following the publication of the paper.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 365.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 426.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 316.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 398.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 441.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 449.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 366.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 384.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 340.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 470.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 399.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 445

In order to request macro data from the BLS API, I generate a BLS series ID for each industry

In [ ]:

# Merge industry titles from industry code map
aiie_code_check = aiie.merge(industry_code_map, left_on="naics", right_on="naics_code", how="outer")
aiie_code_check

Specify constants for analysis:
* Time period: 2019 to 2026
* Seasonal adjustment: Yes, to avoid results being driven by seasonal variation in data.
* Data variable: Total employment in 1000s

In [4]:
START_YEAR = 2019
END_YEAR = 2026
EMP_SERIES = "CES" # National Employment, Hours, and Earnings; seasonally adjusted
EMP_DATA_TYPE = "01" # Employement (Level 1000s)

## Request employment data from BLS

Call the BLS API for employment data by industry at the 4-digit NAICS code level.




Compile the list of series IDs required following this structure:

	Series ID    CEU0800000003
	Positions       Value           Field Name
	1-2             CE              Prefix
	3               U               Seasonal Adjustment Code
	4-11		08000000	Supersector and Industry Codes
	12-13           03              Data Type Code

The industry codes need to be unpacked from the [codes list](https://download.bls.gov/pub/time.series/ce/ce.industry) provided by the BLS. This is contained in a `.tsv` file but is hosted with a faulty extension, so it must be downloaded manually and read as a .tsv.

Some industry codes map onto multiple NAICS codes or vice versa, reflected in the `naics_code` column with either a comma separating the additional final digits, or the NAICS code being split as "part XXXX" across two industry codes.

In order to preserve a one to one mapping to exposure scores I drop these codes.

In [5]:
industry_code_map = (
    pd.read_csv("../data/reference/ce_industry.tsv", delimiter="\t", index_col=False)
    # Selecting display level 5 filters on 4 digit NAICS codes to match with AIOE table
    .query("display_level == 5")
    .filter(["industry_code", "naics_code", "industry_name"])
    .reset_index(drop=True)
)

# Filter out codes without a 1:1 industry code mapping
industry_code_map = industry_code_map[industry_code_map["naics_code"].str.len() == 4]

Define function to generate BLS series ID from industry NAICS code and constants.


In [6]:
def generate_emp_series_id(industry, series=EMP_SERIES, d_type=EMP_DATA_TYPE):
    return f"{series}{industry}{EMP_DATA_TYPE}"

Add column of BLS series IDs to `industry_code_map` by applying `generate_series_id function` and extract list of series IDs. 

Keeping the series IDs in a column of the industry_code_map will allow for converting the series IDs back to their NAICS codes and industry names during data transformation and analysis stages.

In [7]:
industry_code_map["series_id_emp"] = (
    industry_code_map["industry_code"]
    .apply(generate_emp_series_id)
    )
industry_code_map.reset_index(drop=True)

emp_series_ids = industry_code_map["series_id_emp"].to_list()

In [8]:
def bls_api_call(
    series_ids: list,
    api_key=BLS_API_KEY,
    start_year=START_YEAR,
    end_year=END_YEAR,
    timeout = 60
    ):
    ''' 
    Requests data from BLS API from a list of up to 50 BLS series IDs.
    Returns the response in JSON format
    '''
    # headers required by BLS API
    headers = {'Content-type': 'application/json'}
    # Generate request JSON to post to BLS API
    request_data = json.dumps({
            "seriesid":series_ids, 
            "startyear":START_YEAR, 
            "endyear":END_YEAR,
            "registrationkey":BLS_API_KEY
            }
            )
    # Post query to BLS API
    p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=request_data, headers=headers)
    return p.json()


Function to save raw JSON data pulled from BLS API.

In [9]:
def save_raw_data(data, indicator, batch):
    '''
    Saves raw JSON returned by BLS API call into the raw data folder.
    The data will be stored in a folder with the name of the indicator,
    and a batch number will be attached to the end of the file name.
    '''
    folder = Path("..") / "data" / "raw" / indicator.lower()
    
    os.makedirs(folder, exist_ok=True)

    file_path = folder / f"bls_{indicator.lower()}_{batch}.json"

    with open(file_path, mode="w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"Saved to {file_path}")

The BLS API will return up to 50 series IDs per call, so I define a function that will take a long list of series IDs and put them into a list containing batches of 50 series IDs.

In [10]:
def batch_series_ids(series_ids, batch_size=50):
    '''
    Takes a list of series IDs and places them into batches of 50.
    Returns a list containing lists of the batched IDs and prints
    the total number of IDs batched and the number of batches generated 
    as a validation check.
    Different batch sizes can be chosen with parameter batch_size.
    '''
    batch_list = []
    num_series_ids = len(series_ids)
    start = 0
    end = batch_size

    for i in range(num_series_ids // batch_size):
        batch = series_ids[start:end]
        batch_list.append(batch)
        start += batch_size
        end += batch_size
    
    final_batch = num_series_ids - num_series_ids % batch_size
    batch_list.append(
        series_ids[final_batch+1:]
    )

    # check total number of series IDs and batches
    total_batched = sum([len(i) for i in batch_list])
    print(f"{total_batched} series IDs placed into {len(batch_list)} batches")

    return batch_list


## API call: Employment
Call API by looping through batches of series IDs and save raw data as JSONs.

In [39]:
emp_batches = batch_series_ids(emp_series_ids)

for batch in range(len(emp_batches)):
    batch_num = batch+1
    raw_json = bls_api_call(emp_batches[batch])
    save_raw_data(raw_json, indicator="employment", batch=batch_num)

215 series IDs placed into 5 batches
Saved to ../data/raw/employment/bls_employment_1.json
Saved to ../data/raw/employment/bls_employment_2.json
Saved to ../data/raw/employment/bls_employment_3.json
Saved to ../data/raw/employment/bls_employment_4.json
Saved to ../data/raw/employment/bls_employment_5.json


## Save industry code map
Industry code map is saved to the reference folder to enable the series IDs in the raw JSON files to be matched back to their NAICS codes and industry names during data transformation and analysis.

This is done last after all indicators are collected and appended onto the code map, enabling a single file to map to all used series IDs.

In [10]:
industry_code_map.to_csv("../data/reference/industry_code_map.csv", index=False)